# Model Tuning with Vertex AI — Supervised Fine-Tuning

> **Prerequisites:**
> - A GCP project with billing enabled and the Vertex AI API activated
> - A GCS bucket in the same project
> - Run `gcloud auth application-default login` in your terminal before starting
> - Run `gcloud auth application-default set-quota-project YOUR_PROJECT_ID` as well

## Objective

Fine-tune a Gemini foundation model on new data using Vertex AI Supervised Fine-Tuning (SFT). This notebook uses:
- **Vertex AI Tuning** — managed SFT job
- **Google Cloud Storage** — training data storage
- **BigQuery** — source of BBC news dataset
- **google-genai SDK** — unified SDK for both Vertex AI and Gemini API

## Use Case

Generate a suitable **title** for a news **body** from the BBC Fulltext dataset (`bigquery-public-data.bbc_news.fulltext`).  
We fine-tune **gemini-2.5-flash** with SFT to create a model called `bbc-news-summary-tuned`, then compare its output against the base model.

## Install Dependencies

In [1]:
!pip install -q google-genai google-cloud-storage google-cloud-bigquery

## Configuration

Fine-tuning runs on Vertex AI and requires ADC credentials — it cannot use a plain Gemini API key.  
Inference against the base model uses the Gemini API key (cheaper, no billing required).

In [2]:
import os
import getpass
import json
import warnings
import io

import pandas as pd
from google import genai
from google.genai import types
from google.cloud import storage, bigquery

warnings.filterwarnings("ignore")

PROJECT_ID = "mychatbotproject2026" # os.getenv("GOOGLE_CLOUD_PROJECT") or getpass.getpass("Enter your GCP Project ID: ")
REGION = "us-central1"
BUCKET_NAME = "myfinetunedata2026" # os.getenv("GCS_BUCKET_NAME") or getpass.getpass("Enter your GCS bucket name (without gs://): ")

# Vertex AI client — uses Application Default Credentials (ADC)
# Run: gcloud auth application-default login
vertex_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

print(f"Project: {PROJECT_ID}")
print(f"Region:  {REGION}")
print(f"Bucket:  {BUCKET_NAME}")

Project: mychatbotproject2026
Region:  us-central1
Bucket:  myfinetunedata2026


## Prepare Training Data

### Option A — Load from BigQuery (public dataset)
If you don't have a `TRAIN.jsonl` in GCS yet, fetch BBC news data directly from BigQuery.

In [3]:
bq_client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT
    body   AS input_text,
    title  AS output_text
FROM `bigquery-public-data.bbc_news.fulltext`
WHERE body IS NOT NULL AND title IS NOT NULL
LIMIT 500
"""

df = bq_client.query(query).to_dataframe()
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (500, 2)


,input_text,output_text
0,"News Corp, the media company controlled by Aus...",News Corp eyes video games market
1,Millions of the world's poorest textile trade ...,Millions 'to lose textile jobs'
2,The board of Indian conglomerate Reliance has ...,Share boost for feud-hit Reliance
3,German airline Lufthansa may sue federal agenc...,Lufthansa may sue over Bush visit
4,General Motors (GM) saw its net profits fall 3...,European losses hit GM's profits


### Option B — Load existing TRAIN.jsonl from GCS
Skip this cell if you ran Option A above.

In [4]:
# Option B: load from GCS
# storage_client = storage.Client(project=PROJECT_ID)
# data = storage_client.bucket(BUCKET_NAME).blob("TRAIN.jsonl").download_as_bytes()
# df = pd.read_json(io.BytesIO(data), lines=True)
# print(f"Dataset shape: {df.shape}")
# df.head()

## Convert to Gemini SFT Chat Format and Upload to GCS

Gemini SFT requires training data in **JSONL chat format** — each line must have a `messages` array where each message has a `role` and a `parts` list.

In [7]:
local_jsonl = "TRAIN_gemini.jsonl"

with open(local_jsonl, "w") as f:
    for _, row in df.iterrows():
        record = {
            "contents": [
                {
                    "role": "user",
                    "parts": [{"text": f"Summarize this text to generate a title:\n{row['input_text']}"}]
                },
                {
                    "role": "model",
                    "parts": [{"text": row["output_text"]}]
                }
            ]
        }
        f.write(json.dumps(record) + "\n")

# Upload to GCS
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
bucket.blob("TRAIN_gemini.jsonl").upload_from_filename(local_jsonl)

TRAIN_GCS_URI = f"gs://{BUCKET_NAME}/TRAIN_gemini.jsonl"
print(f"Uploaded {len(df)} records to {TRAIN_GCS_URI}")

Uploaded 500 records to gs://myfinetunedata2026/TRAIN_gemini.jsonl


## Start Supervised Fine-Tuning Job

This submits an async SFT job on Vertex AI. It typically takes **30–60 minutes** to complete.  
Supported base models: https://cloud.google.com/vertex-ai/generative-ai/docs/models/tune-models

In [8]:
tuning_job = vertex_client.tunings.tune(
    base_model="gemini-2.5-flash",
    training_dataset=types.TuningDataset(
        gcs_uri=TRAIN_GCS_URI,
    ),
    config=types.CreateTuningJobConfig(
        epoch_count=4,
        learning_rate_multiplier=1.0,
        tuned_model_display_name="bbc-news-summary-tuned",
    ),
)

print("Tuning job name:", tuning_job.name)
print("Tuning job state:", tuning_job.state)

Tuning job name: projects/172368468008/locations/us-central1/tuningJobs/2957745497726517248
Tuning job state: JobState.JOB_STATE_PENDING


## Monitor the Tuning Job

Poll until the job reaches a terminal state. Re-run this cell periodically or wait for it to finish.

In [9]:
import time

while True:
    tuning_job = vertex_client.tunings.get(name=tuning_job.name)
    state = str(tuning_job.state)
    print(f"State: {state}")
    if "SUCCEEDED" in state or "FAILED" in state or "CANCELLED" in state:
        break
    time.sleep(60)

print("Final state:", tuning_job.state)

State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING


State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobState.JOB_STATE_RUNNING
State: JobStat

## Predict with the Fine-Tuned Model

After the job succeeds, retrieve the tuned model endpoint and compare against the base model.

In [10]:
tuned_endpoint = tuning_job.tuned_model.endpoint
print("Tuned model endpoint:", tuned_endpoint)

test_input = (
    "Summarize this text to generate a title:\n"
    "Ever noticed how plane seats appear to be getting smaller and smaller? "
    "With increasing numbers of people taking to the skies, some experts are questioning "
    "if having such packed out planes is putting passengers at risk. They say that the "
    "shrinking space on aeroplanes is not only uncomfortable — it's putting our health and "
    "safety in danger. This week, a U.S. consumer advisory group set up by the Department "
    "of Transportation said at a public hearing that while the government is happy to set "
    "standards for animals flying on planes, it doesn't stipulate a minimum amount of space "
    "for humans."
)

# Fine-tuned model inference (via Vertex AI)
tuned_response = vertex_client.models.generate_content(
    model=tuned_endpoint,
    contents=test_input,
)
print("Fine-tuned model:", tuned_response.text)

# Base model inference (also via Vertex AI for fair comparison)
base_response = vertex_client.models.generate_content(
    model="gemini-2.5-flash",
    contents=test_input,
)
print("Base model:", base_response.text)

Tuned model endpoint: projects/172368468008/locations/us-central1/endpoints/7552167688949202944
Fine-tuned model: Shrinking plane seats are 'a health risk'
Base model: Here are a few title options for the text:

**Option 1 (Direct & Comprehensive):**
Shrinking Plane Seats: A Growing Safety Concern

**Option 2 (Emphasizing the Irony):**
Plane Passengers: Less Space Than Pets?

**Option 3 (Focus on Risk):**
Crowded Planes: Health and Safety Risks Emerge

**Option 4 (Concise & Punchy):**
The Squeeze on Planes: Passenger Safety Questioned
